# Task 2 — Preprocessing

This notebook preprocesses the aligned 10% English–Spanish sample created in Task 1.
Every filtering decision is applied to the complete sentence pair so that the parallel
alignment is preserved.

## Preprocessing strategy

The following steps are used:

1. **Remove empty sentence pairs:** if either the English or Spanish side is empty,
   both lines are discarded.
2. **Remove XML/markup lines:** if either stripped line starts with `<`, the complete
   aligned pair is discarded. Such lines represent metadata rather
   than natural-language sentences.
3. **Lowercase both languages:** this reduces unnecessary vocabulary duplication caused
   by capitalization.
4. **Normalize whitespace:** leading and trailing whitespace is removed and consecutive
   whitespace characters are replaced by one regular space.

Punctuation, numbers, stop words, and inflectional forms are intentionally retained.
They carry information that a translation model should learn to reproduce. Stemming and
lemmatization would also prevent the model from generating natural surface forms.

At the end of the notebook, the complete preprocessing pipeline is provided twice:
once for a word-based model and once for a character-based model. These functions can
be copied directly into the Task 3 notebook.

In [1]:
from itertools import zip_longest
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


PROJECT_ROOT = Path.cwd().parent
SAMPLE_DIR = PROJECT_ROOT / "data" / "sampled"
OUTPUT_DIR = PROJECT_ROOT / "data" / "preprocessed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_EN_PATH = SAMPLE_DIR / "europarl_10pct.en"
INPUT_ES_PATH = SAMPLE_DIR / "europarl_10pct.es"

OUTPUT_EN_PATH = OUTPUT_DIR / "europarl_10pct_preprocessed.en"
OUTPUT_ES_PATH = OUTPUT_DIR / "europarl_10pct_preprocessed.es"

print(f"Project root: {PROJECT_ROOT}")
print(f"English input: {INPUT_EN_PATH}")
print(f"Spanish input: {INPUT_ES_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: /home/theohagen/Desktop/nlp-machine-translation
English input: /home/theohagen/Desktop/nlp-machine-translation/data/sampled/europarl_10pct.en
Spanish input: /home/theohagen/Desktop/nlp-machine-translation/data/sampled/europarl_10pct.es
Output directory: /home/theohagen/Desktop/nlp-machine-translation/data/preprocessed


/home/theohagen/miniforge3/envs/nlp-mt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Define the preprocessing rules

In [2]:
def is_markup_line(text: str) -> bool:
    """Return True for XML/markup lines as described in the assignment."""
    return text.lstrip().startswith("<")


def normalize_sentence(text: str) -> str:
    """Lowercase text and normalize all whitespace to single spaces."""
    return " ".join(text.strip().lower().split())

## 2. Preprocess the aligned corpus

Temporary files are written first and renamed only after the complete corpus has been
processed successfully. This prevents incomplete output files from being mistaken for
finished results if execution is interrupted.

In [3]:
TEMP_EN_PATH = OUTPUT_EN_PATH.with_suffix(OUTPUT_EN_PATH.suffix + ".tmp")
TEMP_ES_PATH = OUTPUT_ES_PATH.with_suffix(OUTPUT_ES_PATH.suffix + ".tmp")

statistics = {
    "input_pairs": 0,
    "removed_empty_pairs": 0,
    "removed_markup_pairs": 0,
    "retained_pairs": 0,
    "changed_english_sentences": 0,
    "changed_spanish_sentences": 0,
}

changed_examples = []
max_examples = 5

try:
    with INPUT_EN_PATH.open("r", encoding="utf-8", errors="replace") as en_file, \
         INPUT_ES_PATH.open("r", encoding="utf-8", errors="replace") as es_file, \
         TEMP_EN_PATH.open("w", encoding="utf-8", newline="\n") as output_en, \
         TEMP_ES_PATH.open("w", encoding="utf-8", newline="\n") as output_es:

        aligned_lines = zip_longest(en_file, es_file, fillvalue=None)

        for line_number, (en_line, es_line) in enumerate(
            tqdm(aligned_lines, desc="Preprocessing aligned sentence pairs"),
            start=1,
        ):
            if en_line is None or es_line is None:
                raise ValueError(
                    "The English and Spanish sample files contain different "
                    f"numbers of lines. Alignment failed near line {line_number}."
                )

            statistics["input_pairs"] += 1

            raw_en = en_line.rstrip("\r\n")
            raw_es = es_line.rstrip("\r\n")

            # Remove the complete aligned pair if either side is empty.
            if not raw_en.strip() or not raw_es.strip():
                statistics["removed_empty_pairs"] += 1
                continue

            # Remove the complete pair if either line is corpus markup.
            if is_markup_line(raw_en) or is_markup_line(raw_es):
                statistics["removed_markup_pairs"] += 1
                continue

            clean_en = normalize_sentence(raw_en)
            clean_es = normalize_sentence(raw_es)

            if clean_en != raw_en:
                statistics["changed_english_sentences"] += 1
            if clean_es != raw_es:
                statistics["changed_spanish_sentences"] += 1

            if (
                len(changed_examples) < max_examples
                and (clean_en != raw_en or clean_es != raw_es)
            ):
                changed_examples.append(
                    {
                        "English before": raw_en,
                        "English after": clean_en,
                        "Spanish before": raw_es,
                        "Spanish after": clean_es,
                    }
                )

            output_en.write(clean_en + "\n")
            output_es.write(clean_es + "\n")
            statistics["retained_pairs"] += 1

    TEMP_EN_PATH.replace(OUTPUT_EN_PATH)
    TEMP_ES_PATH.replace(OUTPUT_ES_PATH)

except Exception:
    TEMP_EN_PATH.unlink(missing_ok=True)
    TEMP_ES_PATH.unlink(missing_ok=True)
    raise

print(f"Saved English corpus: {OUTPUT_EN_PATH}")
print(f"Saved Spanish corpus: {OUTPUT_ES_PATH}")

Preprocessing aligned sentence pairs: 196573it [00:00, 240943.06it/s]

Saved English corpus: /home/theohagen/Desktop/nlp-machine-translation/data/preprocessed/europarl_10pct_preprocessed.en
Saved Spanish corpus: /home/theohagen/Desktop/nlp-machine-translation/data/preprocessed/europarl_10pct_preprocessed.es


## 3. Summarize the preprocessing results

In [4]:
removed_pairs = (
    statistics["removed_empty_pairs"]
    + statistics["removed_markup_pairs"]
)

statistics["removed_pairs_total"] = removed_pairs
statistics["retention_rate_percent"] = (
    100.0 * statistics["retained_pairs"] / statistics["input_pairs"]
    if statistics["input_pairs"]
    else 0.0
)

summary = pd.Series(statistics, name="value")
summary

input_pairs                  196573.000000
removed_empty_pairs             478.000000
removed_markup_pairs              0.000000
retained_pairs               196095.000000
changed_english_sentences    195425.000000
changed_spanish_sentences    195402.000000
removed_pairs_total             478.000000
retention_rate_percent           99.756833
Name: value, dtype: float64

In [5]:
if changed_examples:
    display(pd.DataFrame(changed_examples))
else:
    print("No sentence required lowercasing or whitespace normalization.")

,English before,English after,Spanish before,Spanish after
0,"Please rise, then, for this minute' s silence.","please rise, then, for this minute' s silence.",Invito a todos a que nos pongamos de pie para ...,invito a todos a que nos pongamos de pie para ...
1,The Cunha report on multiannual guidance progr...,the cunha report on multiannual guidance progr...,El informe Cunha sobre los programas de direcc...,el informe cunha sobre los programas de direcc...
2,"Now, however, he is to go before the courts on...","now, however, he is to go before the courts on...","Sin embargo, sucede que va a ser acusado de nu...","sin embargo, sucede que va a ser acusado de nu..."
3,"Yes, Mrs Schroedter, I shall be pleased to loo...","yes, mrs schroedter, i shall be pleased to loo...","Sí, señora Schroedter, de buena gana voy a exa...","sí, señora schroedter, de buena gana voy a exa..."
4,This commitment is important because the Commi...,this commitment is important because the commi...,Tiene importancia este compromiso en la medida...,tiene importancia este compromiso en la medida...


## Output for Task 3

The cleaned parallel corpus is stored in:

- `data/preprocessed/europarl_10pct_preprocessed.en`
- `data/preprocessed/europarl_10pct_preprocessed.es`

The preprocessing has already removed invalid pairs, converted both languages to
lowercase, and normalized whitespace. The following functions therefore only load
the aligned preprocessed files and apply the tokenization required by the word-based
or character-based model.

## 4. Load and tokenize the preprocessed corpus

Both functions accept the paths of the two preprocessed files and return a list
of aligned token pairs:

```python
[
    (english_tokens_1, spanish_tokens_1),
    (english_tokens_2, spanish_tokens_2),
    ...
]
```

`max_pairs` is only for testing.



In [6]:
import re
from itertools import zip_longest
from pathlib import Path

WORD_PATTERN = re.compile(
    r"\w+(?:['’]\w+)*|[^\w\s]",
    flags=re.UNICODE,
)


def load_word_tokenized_pairs(
    english_path,
    spanish_path,
    max_pairs=None,
):
    """Load aligned preprocessed files and apply word tokenization."""
    tokenized_pairs = []

    with Path(english_path).open("r", encoding="utf-8") as en_file, \
         Path(spanish_path).open("r", encoding="utf-8") as es_file:

        for line_number, (en_line, es_line) in enumerate(
            zip_longest(en_file, es_file, fillvalue=None),
            start=1,
        ):
            if en_line is None or es_line is None:
                raise ValueError(
                    "The preprocessed files contain different numbers "
                    f"of lines near line {line_number}."
                )

            en_tokens = WORD_PATTERN.findall(
                en_line.rstrip("\r\n")
            )
            es_tokens = WORD_PATTERN.findall(
                es_line.rstrip("\r\n")
            )
            tokenized_pairs.append((en_tokens, es_tokens))

            if (
                max_pairs is not None
                and len(tokenized_pairs) >= max_pairs
            ):
                break

    return tokenized_pairs


def load_character_tokenized_pairs(
    english_path,
    spanish_path,
    max_pairs=None,
):
    """Load aligned preprocessed files and apply character tokenization."""
    tokenized_pairs = []

    with Path(english_path).open("r", encoding="utf-8") as en_file, \
         Path(spanish_path).open("r", encoding="utf-8") as es_file:

        for line_number, (en_line, es_line) in enumerate(
            zip_longest(en_file, es_file, fillvalue=None),
            start=1,
        ):
            if en_line is None or es_line is None:
                raise ValueError(
                    "The preprocessed files contain different numbers "
                    f"of lines near line {line_number}."
                )

            en_characters = list(en_line.rstrip("\r\n"))
            es_characters = list(es_line.rstrip("\r\n"))
            tokenized_pairs.append(
                (en_characters, es_characters)
            )

            if (
                max_pairs is not None
                and len(tokenized_pairs) >= max_pairs
            ):
                break

    return tokenized_pairs

## 5. Test both tokenizers

Only three pairs are loaded here so that the notebook can display the output
without materializing the full character-tokenized corpus in memory.

In [7]:
word_pairs_preview = load_word_tokenized_pairs(
    "../data/preprocessed/europarl_10pct_preprocessed.en",
    "../data/preprocessed/europarl_10pct_preprocessed.es",
    max_pairs=3,
)

character_pairs_preview = load_character_tokenized_pairs(
    "../data/preprocessed/europarl_10pct_preprocessed.en",
    "../data/preprocessed/europarl_10pct_preprocessed.es",
    max_pairs=3,
)

print("First word-tokenized pair:")
print("English:", word_pairs_preview[0][0])
print("Spanish:", word_pairs_preview[0][1])

print("\nFirst character-tokenized pair:")
print("English:", character_pairs_preview[0][0])
print("Spanish:", character_pairs_preview[0][1])

First word-tokenized pair:
English: ['please', 'rise', ',', 'then', ',', 'for', 'this', 'minute', "'", 's', 'silence', '.']
Spanish: ['invito', 'a', 'todos', 'a', 'que', 'nos', 'pongamos', 'de', 'pie', 'para', 'guardar', 'un', 'minuto', 'de', 'silencio', '.']

First character-tokenized pair:
English: ['p', 'l', 'e', 'a', 's', 'e', ' ', 'r', 'i', 's', 'e', ',', ' ', 't', 'h', 'e', 'n', ',', ' ', 'f', 'o', 'r', ' ', 't', 'h', 'i', 's', ' ', 'm', 'i', 'n', 'u', 't', 'e', "'", ' ', 's', ' ', 's', 'i', 'l', 'e', 'n', 'c', 'e', '.']
Spanish: ['i', 'n', 'v', 'i', 't', 'o', ' ', 'a', ' ', 't', 'o', 'd', 'o', 's', ' ', 'a', ' ', 'q', 'u', 'e', ' ', 'n', 'o', 's', ' ', 'p', 'o', 'n', 'g', 'a', 'm', 'o', 's', ' ', 'd', 'e', ' ', 'p', 'i', 'e', ' ', 'p', 'a', 'r', 'a', ' ', 'g', 'u', 'a', 'r', 'd', 'a', 'r', ' ', 'u', 'n', ' ', 'm', 'i', 'n', 'u', 't', 'o', ' ', 'd', 'e', ' ', 's', 'i', 'l', 'e', 'n', 'c', 'i', 'o', '.']
